<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/Nikhitha/student_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy pandas scikit-learn streamlit joblib pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.4 MB/s eta 0:00:00


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amrmaree/student-performance-prediction")

print("Path to dataset files:", path)

100%|██████████| 10.7k/10.7k [00:00<00:00, 4.75MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/amrmaree/student-performance-prediction/versions/2


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
import joblib
import os

# 1) Load Kaggle dataset (adjust path/filename if needed)
path = "/root/.cache/kagglehub/datasets/amrmaree/student-performance-prediction/versions/2"
csv_name = "student_performance_dataset.csv"   # change if os.listdir(path) shows a different name
csv_path = os.path.join(path, csv_name)

df = pd.read_csv(csv_path)

# 2) Rename Kaggle columns -> your project columns
#   FIRST: run df.head() once in another cell and check the exact column names,
#   THEN edit the keys on the left side below to match.
df = df.rename(columns={
    "Study_Hours_per_Week": "study_hours",
    "Attendance_Rate": "attendance",
    "Past_Exam_Scores": "internal_marks",
    "Final_Exam_Score": "final_marks",
    "Pass_Fail": "pass_fail"
})

# 3) Convert Pass/Fail text to 1/0 if needed
if df["pass_fail"].dtype == "object":
    df["pass_fail"] = df["pass_fail"].map({"Pass": 1, "Fail": 0})

# 4) Same training code as before
X = df[["study_hours", "attendance", "internal_marks"]]
y_class = df["pass_fail"]
y_reg = df["final_marks"]

X_train, X_test, y_class_train, y_class_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42
)
X_train_reg, X_test_reg, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_reg_scaled = scaler.transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_class_train)

lin_model = LinearRegression()
lin_model.fit(X_train_reg_scaled, y_reg_train)

os.makedirs("models", exist_ok=True)
joblib.dump(log_model, "models/logistic_pass_fail.pkl")
joblib.dump(lin_model, "models/linear_marks.pkl")
joblib.dump(scaler, "models/scaler.pkl")

print("Models saved.")

Models saved.


In [ ]:
print(df.columns)


Index(['Student_ID', 'Gender', 'study_hours', 'attendance', 'Past_Exam_Scores',
       'Parental_Education_Level', 'Internet_Access_at_Home',
       'Extracurricular_Activities', 'final_marks', 'pass_fail'],
      dtype='object')


In [8]:
%%writefile app.py
import streamlit as st
import numpy as np
import joblib

log_model = joblib.load("models/logistic_pass_fail.pkl")
lin_model = joblib.load("models/linear_marks.pkl")
scaler = joblib.load("models/scaler.pkl")

st.title("Student Performance Predictor (Colab)")

study_hours = st.number_input("Study hours per day", min_value=0.0, max_value=16.0, value=3.0, step=0.5)
attendance = st.slider("Attendance (%)", min_value=0, max_value=100, value=80, step=1)
internal_marks = st.number_input("Internal marks", min_value=0.0, max_value=30.0, value=20.0, step=1.0)

if st.button("Predict"):
    features = np.array([[study_hours, attendance, internal_marks]])
    features_scaled = scaler.transform(features)

    prob_pass = log_model.predict_proba(features_scaled)[0][1]
    label = "Pass" if prob_pass >= 0.5 else "Fail"

    predicted_marks = lin_model.predict(features_scaled)[0]

    st.subheader("Results")
    st.write(f"Predicted status: **{label}** (probability of pass: {prob_pass:.2f})")
    st.write(f"Predicted final marks: **{predicted_marks:.1f} / 100**")


Writing app.py


In [9]:
!ngrok config add-authtoken 37C0DWHKYadDMfMlzDS4XV5bbmb_5YhjVTaaEADcBYAHKgWB3


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [10]:
from pyngrok import ngrok

ngrok.set_auth_token("37C0DWHKYadDMfMlzDS4XV5bbmb_5YhjVTaaEADcBYAHKgWB3")


In [12]:
!pip install -q streamlit pyngrok

In [13]:
from pyngrok import ngrok
ngrok.set_auth_token("37CoP8RtT1rFbB4k7xZNs9BaLf9_7y5BEyEnP211vwYwBNWBj")


In [14]:
import subprocess
import time
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Start Streamlit
process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ]
)

# Wait for Streamlit to start
time.sleep(6)

# Create ngrok tunnel
public_url = ngrok.connect(addr=8501, proto="http")
print("🚀 Streamlit Public URL:", public_url)


🚀 Streamlit Public URL: NgrokTunnel: "https://unradiated-taenidial-jutta.ngrok-free.dev" -> "http://localhost:8501"
